# SimpleTM ++ Multi-Dataset Evaluation for SWT and FFT
This notebook runs `SimpleTM_SWT` and `SimpleTM_FFT` across multiple datasets. It will systematically loop through all ETT, Weather, Traffic, and Electricity datasets, execute the models, and zip the final checkpoints and results for easy download.

In [1]:
!pip install PyWavelets gdown --quiet
import torch
import os
import shutil

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

PyTorch version: 2.9.0+cu126
CUDA available: True


In [2]:
# 1. Clone Repository (Update to your fork if needed!)
REPO_URL = "https://github.com/agamswami/SimpleTMG.git"
REPO_DIR = "/kaggle/working/SimpleTM"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

# Patch Numpy 2.0 Compatibility 
!sed -i 's/np\.Inf/np.inf/g' utils/tools.py

Cloning into '/kaggle/working/SimpleTM'...
remote: Enumerating objects: 57, done.
remote: Counting objects: 100% (57/57), done.
remote: Compressing objects: 100% (40/40), done.
remote: Total 57 (delta 12), reused 57 (delta 12), pack-reused 0 (from 0)
Receiving objects: 100% (57/57), 7.08 MiB | 23.47 MiB/s, done.
Resolving deltas: 100% (12/12), done.
Working directory: /kaggle/working/SimpleTM


In [3]:
# 2. Download Datasets
FILE_ID = "1hTpUrhe1yEIGa9mCiGxM5rDyzlYKAnyx"
OUTPUT_PATH = "/kaggle/working/dataset.zip"
DATASET_DIR = "./dataset"

if not os.path.exists(DATASET_DIR):
    !gdown https://drive.google.com/uc?id={FILE_ID} -O {OUTPUT_PATH}
    import zipfile
    os.makedirs(DATASET_DIR, exist_ok=True)
    with zipfile.ZipFile(OUTPUT_PATH, 'r') as zip_ref:
        zip_ref.extractall(DATASET_DIR)
    print("Dataset extracted successfully.")
else:
    print("Dataset already exists.")

Downloading...
From (original): https://drive.google.com/uc?id=1hTpUrhe1yEIGa9mCiGxM5rDyzlYKAnyx
From (redirected): https://drive.google.com/uc?id=1hTpUrhe1yEIGa9mCiGxM5rDyzlYKAnyx&confirm=t&uuid=5218928a-8a0c-4db6-8cc0-c4ab45eedf9e
To: /kaggle/working/dataset.zip
100%|████████████████████████████████████████| 171M/171M [00:01<00:00, 95.7MB/s]
Dataset extracted successfully.


In [4]:
# 3. Define Dataset Configurations
# Important settings:
# - enc_in: Number of variates for the dataset.
# - d_model/d_ff: Kept at 32 for smaller datasets, but raised for massive ones like Traffic to prevent extreme bottlenecks.

datasets = [
    # {name, data, root, path, enc_in, d_model, d_ff}
    {"name": "ETTh1", "data": "ETTh1", "root": "./dataset/SimpleTM_datasets/ETT-small", "path": "ETTh1.csv", "enc_in": 7, "d_model": 32, "d_ff": 32},
    {"name": "ETTh2", "data": "ETTh2", "root": "./dataset/SimpleTM_datasets/ETT-small", "path": "ETTh2.csv", "enc_in": 7, "d_model": 32, "d_ff": 32},
    {"name": "ETTm1", "data": "ETTm1", "root": "./dataset/SimpleTM_datasets/ETT-small", "path": "ETTm1.csv", "enc_in": 7, "d_model": 32, "d_ff": 32},
    {"name": "ETTm2", "data": "ETTm2", "root": "./dataset/SimpleTM_datasets/ETT-small", "path": "ETTm2.csv", "enc_in": 7, "d_model": 32, "d_ff": 32},
    {"name": "weather", "data": "custom", "root": "./dataset/SimpleTM_datasets/weather", "path": "weather.csv", "enc_in": 21, "d_model": 32, "d_ff": 32},
    
    # Optional: You can uncomment these massive datasets below if you have the computation time.
    # {"name": "electricity", "data": "custom", "root": "./dataset/SimpleTM_datasets/electricity", "path": "electricity.csv", "enc_in": 321, "d_model": 128, "d_ff": 128},
    # {"name": "traffic", "data": "custom", "root": "./dataset/SimpleTM_datasets/traffic", "path": "traffic.csv", "enc_in": 862, "d_model": 256, "d_ff": 256},
]

# The two models to ablate
models = ["SimpleTM_SWT", "SimpleTM_FFT"]

# Directory to store everything safely outside the repo so we don't accidentally wipe it
OUTPUT_DIR = "/kaggle/working/SimpleTM_Results"
os.makedirs(OUTPUT_DIR, exist_ok=True)


In [5]:
# 4. Automate the Training Loop over ALL datasets
import subprocess

for ds in datasets:
    print(f"\n" + "="*80)
    print(f"\n🔥 STARTING EXPERIMENT: Dataset {ds['name']} (Variates: {ds['enc_in']})")
    print("\n" + "="*80)
    
    for model in models:
        print(f"\n>>> Sub-experiment: Training {model} on {ds['name']} <<<\n")
        
        # We construct the terminal command as a list for subprocess
        cmd = [
            "python", "-u", "run.py",
            "--is_training", "1",
            "--model", model,
            "--model_id", f"{ds['name']}_{model}",
            "--data", ds["data"],
            "--root_path", ds["root"],
            "--data_path", ds["path"],
            "--features", "M",
            "--seq_len", "96",
            "--pred_len", "96",
            "--e_layers", "1",
            "--d_model", str(ds["d_model"]),
            "--d_ff", str(ds["d_ff"]),
            "--enc_in", str(ds["enc_in"]),
            "--dec_in", str(ds["enc_in"]),
            "--c_out", str(ds["enc_in"]), # Used for plotting
            "--wv", "db1",
            "--m", "3",
            "--alpha", "1.0",
            "--learning_rate", "0.0001",
            "--batch_size", "32",
            "--train_epochs", "10",
            "--patience", "3",
            "--num_workers", "2",
            "--checkpoints", f"{OUTPUT_DIR}/checkpoints/", # Send model weights directly to output
            "--fix_seed", "2025"
        ]
        
        # Execute the python script and stream its logs to the Jupyter Notebook output
        try:
            process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
            for line in process.stdout:
                print(line, end="")
            process.wait()
            
            if process.returncode == 0:
                print(f"\n✅ SUCCESS: Finished {model} on {ds['name']}")
            else:
                print(f"\n❌ ERROR: {model} failed on {ds['name']} (exit code {process.returncode})")
        except Exception as e:
            print(f"\n❌ FATAL: Exception running {model} on {ds['name']}: {e}")
        
    print(f"\n" + "-"*80)
    print(f"Finished evaluating both models on {ds['name']}")
    print("-" + "-"*80)

# Save the parsed results text file to our safe output directory
if os.path.exists('result_long_term_forecast.txt'):
    shutil.copy('result_long_term_forecast.txt', os.path.join(OUTPUT_DIR, 'result_long_term_forecast.txt'))
    print("Copied result_long_term_forecast.txt to output directory.")




🔥 STARTING EXPERIMENT: Dataset ETTh1 (Variates: 7)


>>> Sub-experiment: Training SimpleTM_SWT on ETTh1 <<<

Args in experiment:
Namespace(is_training=1, model_id='ETTh1_SimpleTM_SWT', model='SimpleTM_SWT', data='ETTh1', root_path='./dataset/SimpleTM_datasets/ETT-small', data_path='ETTh1.csv', features='M', target='OT', freq='h', checkpoints='/kaggle/working/SimpleTM_Results/checkpoints/', seq_len=96, label_len=0, pred_len=96, enc_in=7, dec_in=7, c_out=7, n_heads=8, d_layers=1, moving_avg=25, factor=1, distil=True, dropout=0.1, geomattn_dropout=0.5, embed='timeF', activation='gelu', do_predict=False, num_workers=2, itr=1, train_epochs=10, batch_size=32, patience=3, learning_rate=0.0001, des='test', loss='MSE', lradj='type1', pct_start=0.2, use_amp=False, use_gpu=True, gpu=0, use_multi_gpu=False, devices='0,1,2,3', exp_name='MTSF', channel_independence=False, inverse=False, class_strategy='projection', target_root_path='./data/electricity/', target_data_path='electricity.csv', efficie

In [6]:
# 5. Summary & Visualization Table
# This cell parses the text file outputs into a clean Pandas dataframe
import pandas as pd
import re

results_file = os.path.join(OUTPUT_DIR, 'result_long_term_forecast.txt')

if os.path.exists(results_file):
    with open(results_file, 'r') as f:
        content = f.read()
    
    # Split by double newline (which run.py naturally outputs between runs)
    entries = content.strip().split('\n\n')
    results = []
    
    for entry in entries:
        lines = entry.strip().split('\n')
        if len(lines) >= 2:
            setting = lines[0].strip()
            metrics_line = lines[-1].strip() # Usually last line is the numerical score
            
            # Simple regex to see which model was run
            model_match = re.search(r'(SimpleTM_(?:SWT|FFT))', setting)
            model_type = model_match.group(1) if model_match else "SimpleTM"
            
            # Since our model_id is {name}_{model}, we can grab the dataset name
            dataset_name = setting.split('_')[0]
            
            # Grab actual metrics
            mse_match = re.search(r'mse:([\d.]+)', metrics_line)
            mae_match = re.search(r'mae:([\d.]+)', metrics_line)
            
            if mse_match and mae_match:
                results.append({
                    'Dataset': dataset_name,
                    'Model': model_type,
                    'MSE': float(mse_match.group(1)),
                    'MAE': float(mae_match.group(1)),
                })
    
    if results:
        df = pd.DataFrame(results)
        print("\n" + "="*80)
        print("                        FINAL COMPARISON TABLE")
        print("="*80)
        
        # We pivot it so models are columns, datasets are rows
        pivot_df = df.pivot(index='Dataset', columns='Model', values=['MSE', 'MAE'])
        print(pivot_df)
        print("="*80)
        
        # Save to CSV for downloading
        pivot_df.to_csv(os.path.join(OUTPUT_DIR, "final_metrics.csv"))
        print(f"Saved exact final table to: {os.path.join(OUTPUT_DIR, 'final_metrics.csv')}")
    else:
        print("No valid metrics found in result file.")
else:
    print("Result file not formed properly/doesn't exist.")



                        FINAL COMPARISON TABLE
                 MSE                       MAE             
Model   SimpleTM_FFT SimpleTM_SWT SimpleTM_FFT SimpleTM_SWT
Dataset                                                    
ETTh1       0.457091     0.451259     0.449469     0.446162
ETTh2       0.313288     0.312984     0.360546     0.360527
ETTm1       0.363031     0.349717     0.382873     0.379085
ETTm2       0.186988     0.186986     0.271123     0.271967
weather     0.197691     0.192925     0.238129     0.233921
Saved exact final table to: /kaggle/working/SimpleTM_Results/final_metrics.csv


In [7]:
# 6. Prepare Zip File for Final Download
print("Packing up all logs, weights, and results...")
shutil.make_archive("/kaggle/working/SimpleTM_All_Experiments", 'zip', OUTPUT_DIR)
print("\n✅ DONE! You can now download 'SimpleTM_All_Experiments.zip' from the Kaggle Output pane.")
print("It contains weights, PDFs of plots, and the CSV spreadsheet of your exact MSE/MAE numbers.")


Packing up all logs, weights, and results...

✅ DONE! You can now download 'SimpleTM_All_Experiments.zip' from the Kaggle Output pane.
It contains weights, PDFs of plots, and the CSV spreadsheet of your exact MSE/MAE numbers.
